# Adversarial attacks on MNIST — in-class demo and take-home exercises

**How this works.** Sections 1–4 are the demo we run together in class: you can follow along,
and you should re-run them at home before doing the exercises. **Sections 5 and 6 are the
take-home exercises** — the part you are asked to do on your own: in Section 5 you complete one
line of the attack and run it, first on the network from the blackboard and then on MNIST;
Section 6 builds on it. Nothing is submitted: bring the notebook with the cells executed and the
answers written in, and we discuss it in the next class.

Everything runs on CPU in about 5 minutes: the models are **already trained** (no training here)
and we evaluate on **500 test images**.

Models (folder `models/`):
- `cnn` — a standard convolutional network
- `dnn` — a 2-layer fully connected network (used for the transfer attack)
- `cnn_robust` — the same CNN, trained adversarially with PGD

All perturbations are in the $\ell_\infty$ norm: $\|\eta\|_\infty \le \varepsilon$. $C_p(x_0,\varepsilon)$ is the allowed set, $f$ the model, $y$ the true class.

Partly inspired by the tutorial by Kolter and Madry, *Adversarial Robustness: Theory and Practice*,
https://adversarial-ml-tutorial.org/.

**Setup:** follow `README.md` (5 minutes) and run `python check_setup.py` before you start. Open
this notebook from the folder that contains `models/` and `data/`.


In [ ]:
# libraries; fixed seed so that runs are reproducible
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
torch.manual_seed(0)

# the two architectures: a CNN (cnn, cnn_robust) and a 2-layer fully connected net (dnn)
class Flatten(nn.Module):
    def forward(self, x): return x.view(x.shape[0], -1)

def make_cnn():
    return nn.Sequential(nn.Conv2d(1,32,3,padding=1), nn.ReLU(),
        nn.Conv2d(32,32,3,padding=1,stride=2), nn.ReLU(),
        nn.Conv2d(32,64,3,padding=1), nn.ReLU(),
        nn.Conv2d(64,64,3,padding=1,stride=2), nn.ReLU(),
        Flatten(), nn.Linear(7*7*64,100), nn.ReLU(), nn.Linear(100,10))

# load the pre-trained weights; we never train, so the weights are frozen
def load(model, path):
    model.load_state_dict(torch.load(path, map_location="cpu")); return model.eval()

cnn = load(make_cnn(), "models/model_cnn.pt")
cnn_robust = load(make_cnn(), "models/model_cnn_robust.pt")
dnn = load(nn.Sequential(Flatten(), nn.Linear(784,200), nn.ReLU(), nn.Linear(200,10)),
           "models/model_dnn_2.pt")
for m in (cnn, cnn_robust, dnn):
    for p in m.parameters(): p.requires_grad_(False)

# the first 500 MNIST test images, pixels in [0,1]: X is [500,1,28,28], y the labels
test = datasets.MNIST("./data", train=False, download=True, transform=transforms.ToTensor())
X = torch.stack([test[i][0] for i in range(500)])
y = torch.tensor([test[i][1] for i in range(500)])

# fraction of images whose highest score is the true class
def accuracy(model, X, y):
    with torch.no_grad(): return (model(X).argmax(1) == y).float().mean().item()

print("clean accuracy -> cnn: %.3f  dnn: %.3f  cnn_robust: %.3f" %
      (accuracy(cnn,X,y), accuracy(dnn,X,y), accuracy(cnn_robust,X,y)))


In [ ]:
# the first n images, titled with the prediction: green = correct, red = wrong
def show(X, y, model, n=10, title=""):
    with torch.no_grad(): p = model(X[:n]).argmax(1)
    f, ax = plt.subplots(1, n, figsize=(n*1.2, 1.7))
    for i in range(n):
        ax[i].imshow(X[i,0], cmap="gray", vmin=0, vmax=1); ax[i].axis("off")
        ax[i].set_title(str(p[i].item()), color="g" if p[i]==y[i] else "r")
    f.suptitle(title); plt.show()


## 1. FGSM

A single step along the sign of the gradient of the loss:

$$x' = x + \varepsilon \cdot \mathrm{sign}\big(\nabla_x \mathrm{loss}_y(x)\big)$$

where $\mathrm{loss}_y$ is the cross-entropy of the true class $y$. The function below returns the
perturbation $\eta = x' - x$, clipped so that $x'$ stays in $[0,1]$.


In [ ]:
# FGSM: one step of size eps along the sign of the gradient; returns eta, not the image
def fgsm(model, X, y, eps):
    eta = torch.zeros_like(X, requires_grad=True)
    nn.CrossEntropyLoss()(model(X + eta), y).backward()
    return (X + eps * eta.grad.sign()).clamp(0, 1) - X   # stay inside [0,1]

show(X, y, cnn, title="originals")
show(X + fgsm(cnn, X, y, 0.1), y, cnn, title="FGSM eps=0.1")


In [ ]:
# accuracy of the cnn under FGSM, as eps grows
eps_list = [0, 0.05, 0.1, 0.2, 0.3]
acc_fgsm = [accuracy(cnn, X + fgsm(cnn, X, y, e), y) for e in eps_list]
plt.plot(eps_list, acc_fgsm, "o-"); plt.xlabel("epsilon"); plt.ylabel("accuracy")
plt.title("CNN under FGSM"); plt.grid(); plt.show()


**Question:** at which $\varepsilon$ does the image still look identical to you, while the accuracy
has already collapsed?


## 2. PGD

FGSM, iterated: several small steps of size $\alpha$, each one **projected** back onto the
$\ell_\infty$ ball of radius $\varepsilon$. Step $\alpha = \varepsilon/4$, as in the slides. With
`random_start=True` we start from a random point inside the ball. The pseudocode in the slides
always starts from random noise; here the default is `random_start=False`, so that we can compare
the two.

This one is given to you in full. Read it: the attack you write yourself, in Section 5, is the same loop with the loss replaced by the margin.


In [ ]:
# PGD: repeated small steps, each followed by the projection onto the eps-ball and onto [0,1]
def pgd(model, X, y, eps, alpha=None, iters=40, random_start=False):
    alpha = alpha or eps / 4               # as in the slides: alpha ~ eps/4
    # start at x, or at a random point of the ball
    eta = (torch.rand_like(X) * 2 * eps - eps) if random_start else torch.zeros_like(X)
    eta = ((X + eta).clamp(0, 1) - X).requires_grad_(True)
    for _ in range(iters):
        loss = nn.CrossEntropyLoss()(model(X + eta), y)
        loss.backward()
        # step along the sign of the gradient, then project onto the ball and onto [0,1]
        d = eta + alpha * eta.grad.sign()
        d = d.clamp(-eps, eps)
        d = (X + d).clamp(0, 1) - X
        # keep the point, drop the graph: the next gradient is taken here
        eta = d.detach().requires_grad_(True)
    return eta.detach()


In [ ]:
# --- What the sanity checks look like. The ones that matter are in Section 5. ---
# PGD on 50 images, then two checks: eta inside the ball, image inside [0,1]
_X, _y = X[:50], y[:50]
_d = pgd(cnn, _X, _y, eps=0.1, iters=20)

assert _d.abs().max().item() <= 0.1 + 1e-6,                      "eta outside the l_inf ball"
assert (_X + _d).min().item() >= -1e-6 and (_X + _d).max().item() <= 1 + 1e-6, "image outside [0,1]"
print("PGD behaves: max |eta| = %.3f, accuracy %.3f -> %.3f"
      % (_d.abs().max(), accuracy(cnn, _X, _y), accuracy(cnn, _X + _d, _y)))


In [ ]:
# FGSM against PGD with more and more iterations, at the same eps
eps = 0.1
print("FGSM           :", accuracy(cnn, X + fgsm(cnn, X, y, eps), y))
for it in (1, 5, 10, 40):
    print(f"PGD {it:>2} iter    :", accuracy(cnn, X + pgd(cnn, X, y, eps, iters=it), y))
print("PGD 40 + random:", accuracy(cnn, X + pgd(cnn, X, y, eps, iters=40, random_start=True), y))


In [ ]:
# accuracy under PGD-20 for every eps, next to the FGSM curve
acc_pgd = [accuracy(cnn, X + pgd(cnn, X, y, e, iters=20), y) for e in eps_list]
plt.plot(eps_list, acc_fgsm, "o-", label="FGSM"); plt.plot(eps_list, acc_pgd, "s-", label="PGD-20")
plt.xlabel("epsilon"); plt.ylabel("accuracy"); plt.legend(); plt.grid(); plt.show()


**Questions:** is PGD always stronger than FGSM at the same $\varepsilon$? What happens with a
single iteration, and why? Does the random start help on the standard CNN?


## 3. Transfer

We build the perturbation on model A (`dnn`) and apply it to model B (`cnn`), without ever
touching the gradients of B.


In [ ]:
# perturbation built on src, always evaluated on the cnn
eps = 0.2
for name, src in (("dnn  -> cnn (transfer) ", dnn), ("cnn  -> cnn (white-box)", cnn)):
    Xa = X + pgd(src, X, y, eps)
    print(f"{name}: accuracy of cnn = {accuracy(cnn, Xa, y):.3f}")
print("dnn  -> dnn (white-box): accuracy of dnn = %.3f" % accuracy(dnn, X + pgd(dnn, X, y, eps), y))


**Question:** the transfer works without knowing the weights of B. What does that suggest — are
these vulnerabilities a property of the *model*, or of the *data*?


## 4. A robust model

`cnn_robust` was trained *on PGD examples* (Madry's min-max formulation). Let us redo the curve and
compare.


In [ ]:
# the same PGD-20 curve on the adversarially trained cnn
acc_rob = [accuracy(cnn_robust, X + pgd(cnn_robust, X, y, e, iters=20), y) for e in eps_list]
plt.plot(eps_list, acc_pgd, "s-", label="standard cnn")
plt.plot(eps_list, acc_rob, "^-", label="cnn_robust (adv. training)")
plt.xlabel("epsilon (PGD-20)"); plt.ylabel("accuracy"); plt.legend(); plt.grid(); plt.show()
print("eps       :", eps_list)
print("standard  :", [round(a,3) for a in acc_pgd])
print("robust    :", [round(a,3) for a in acc_rob])
print("clean accuracy: standard %.3f | robust %.3f" % (accuracy(cnn, X, y), accuracy(cnn_robust, X, y)))


**Questions:** how much clean accuracy do we pay for robustness? At which $\varepsilon$ does even the
robust model break? And: if a *weaker* attack (say FGSM, or a handful of steps) reported a high
accuracy, would that entitle us to call the model robust? (→ this is the thread that leads to
**verification**.)


## 4b. Sanity checks (*"THE TRAP"* in the slides)

An attack that *fails* proves nothing. Three tests that catch an attack — or a gradient — that is
lying to you:

1. **Does one step beat many?** If FGSM lowers the accuracy *more* than PGD, the gradient is lying.
2. **Does black-box beat white-box?** Whoever knows less must not do better (compare with Section 3).
3. **Unbounded $\varepsilon$:** with a huge budget every attack must succeed. If the accuracy does
   not go to ~0, what is broken is the attack, not the model.


In [ ]:
print(f"{'model':<12} {'FGSM':>6} {'PGD-20':>7} {'PGD-20, eps=1':>14}   (eps=0.1 except the last column)")
# a stronger attack must never do worse than a weaker one, and eps=1 must break everything
for name, m in (("cnn", cnn), ("cnn_robust", cnn_robust)):
    a_f = accuracy(m, X + fgsm(m, X, y, 0.1), y)
    a_p = accuracy(m, X + pgd(m, X, y, 0.1, iters=20), y)
    a_inf = accuracy(m, X + pgd(m, X, y, 1.0, iters=20), y)
    flag = "  <-- FGSM beats PGD: suspicious!" if a_f < a_p - 1e-9 else ""
    print(f"{name:<12} {a_f:>6.3f} {a_p:>7.3f} {a_inf:>14.3f}{flag}")


**Three numbers** : the **clean** accuracy; the
**empirical** robust accuracy (under some attack — an **upper bound**, it only goes down as the
attack gets better); the **certified** robust accuracy (a **lower bound**, computed by a verifier —
and that is the subject of the next block of the course).


In [ ]:
print(f"{'model':<12} {'clean':>7} {'empirical (PGD-20, eps=0.1)':>29} {'certified':>11}")
# clean and empirical robust accuracy; the certified one comes with verification
for name, m in (("cnn", cnn), ("cnn_robust", cnn_robust)):
    print(f"{name:<12} {accuracy(m, X, y):>7.3f} "
          f"{accuracy(m, X + pgd(m, X, y, 0.1, iters=20), y):>29.3f} {'?':>11}")


## 5. Exercise 1 — Falsification: from the blackboard to MNIST

In the slides the property is $f(x) > 0$ for every $x \in C_p(x_0,\varepsilon)$, and the attack looks for **one**
point that violates it: $\exists x \in C_p(x_0,\varepsilon):\ f(x) \le 0$ — a counterexample, which the
machine-learning literature calls an adversarial example. For a classifier, $f$ is the **margin**
$f_y(x) - \max_{j \ne y} f_j(x)$. We minimise $f$ with PGD: if we reach $f \le 0$, the property is
**falsified**.

Write $y^\star = \min \{\, f(x) : x \in C_p(x_0,\varepsilon) \,\}$. The property holds exactly when
$y^\star > 0$, and what PGD returns is an **upper bound on $y^\star$** — never $y^\star$ itself.
Upper bound $\le 0$ means falsified; above $0$ we have learned nothing.

This flavour of PGD is the one used inside verifiers (e.g. alpha-beta-CROWN):

- the loss is the margin of the specification, with a *hinge* (it stops pushing once $f < 0$);
- the start is **uniform over the whole** $C_p(x_0,\varepsilon)$, with many **restarts** run in parallel as a batch;
- the projection is a clamp to the box, the step size decays, and there is an **early stop** as soon
  as a counterexample is found.

(Simplified: we use a sign-gradient step, without an Adam optimiser.)

**Fill in the line marked `FILL_IN`.** It is the same step as in Section 2, with one difference: there we climbed the loss, here we descend the margin.


In [ ]:
import torch.nn.functional as F

def spec_class(z, y):      # classification margin: f(x) = f_y(x) - max_{j!=y} f_j(x)   (y = correct class)
    return z.gather(1, y[:, None])[:, 0] - z.masked_fill(F.one_hot(y, z.shape[1]).bool(), -1e9).max(1)[0]

def spec_score(z, y=None): # a network with a single score f(x): the specification is f(x) > 0
    return z[:, 0]

# PGD on the margin, with random starts and restarts (as inside a verifier)
def pgd_falsify(model, X, y, eps, spec, steps=50, restarts=10, alpha=None, decay=0.99,
                domain=(0., 1.), early_stop=True):
    """Minimise the margin spec(model(x), y) over C_p(x0, eps) = {x: |x-x0|_inf <= eps}, clipped to `domain`.
    Returns (smallest margin found, eta): margin <= 0  =>  counterexample, property falsified."""
    B = X.shape[0]; alpha = alpha or eps / 4
    # every image repeated `restarts` times: all restarts run as one batch
    Xr = X.repeat_interleave(restarts, 0)
    yr = None if y is None else y.repeat_interleave(restarts, 0)
    lo, hi = (Xr - eps).clamp(*domain) - Xr, (Xr + eps).clamp(*domain) - Xr   # the box C, in terms of eta
    eta = (lo + torch.rand_like(Xr) * (hi - lo)).requires_grad_(True)       # uniform start inside C
    # smallest margin found so far for every copy, and the eta that reached it
    best_margin = torch.full((B * restarts,), float("inf")); best_eta = torch.zeros_like(Xr)
    for t in range(steps):
        margin = spec(model(Xr + eta), yr)
        better = margin.detach() < best_margin
        best_margin = torch.where(better, margin.detach(), best_margin)
        best_eta[better] = eta.detach()[better]
        if early_stop and (best_margin.view(B, restarts).min(1)[0] <= 0).all(): break   # early stop
        margin.clamp(min=-1e-5).sum().backward()                               # hinge
        # TODO: one step DOWN the margin, then clamp back into the box C
        eta = torch.min(torch.max(eta - alpha * FILL_IN, lo), hi).detach().requires_grad_(True)
        alpha *= decay
    # for every image, the best of its restarts
    bm, idx = best_margin.view(B, restarts).min(1)
    return bm, best_eta.view(B, restarts, *X.shape[1:])[torch.arange(B), idx]


In [ ]:
# --- Sanity checks. These must pass before you go on. ---
# a few correctly classified images, few restarts, to be fast
_ok = cnn(X[:40]).argmax(1) == y[:40]
_X, _y = X[:40][_ok], y[:40][_ok]
_bm, _eta = pgd_falsify(cnn, _X, _y, 0.1, spec_class, restarts=3)

assert _eta.abs().max().item() <= 0.1 + 1e-6, \
    "eta leaves the l_inf ball: the clamp onto lo/hi is missing or wrong"
assert (_X + _eta).min().item() >= -1e-6 and (_X + _eta).max().item() <= 1 + 1e-6, \
    "the counterexample is not a valid image: the box C already encodes [0,1], check lo and hi"
with torch.no_grad():
    _m0 = spec_class(cnn(_X), _y)
assert _bm.mean().item() < _m0.mean().item(), \
    "the margin did not go DOWN on average: check the SIGN of the step (we minimise f here)"
assert (_bm <= 0).any(), \
    "no counterexample at all at eps=0.1 on the standard cnn: expected about half of them"
print("falsification sanity checks passed - %.0f%% falsified, smallest margin %.2f"
      % (100*(_bm <= 0).float().mean(), _bm.min()))


### 5a. The network from the blackboard

$f(x) = 2\,\mathrm{ReLU}(x_1 - x_2) - \mathrm{ReLU}(x_1 + x_2) + 1.6$, class $+$ iff $f > 0$.
Start at $x^0 = (1, 0.5)$, where $f = 1.1$. Budget $\varepsilon = 0.4$ in $\ell_\infty$, step
$\alpha = \varepsilon/4 = 0.1$, inputs in $[0,1]^2$. First we reproduce the steps we did by hand,
then we let `pgd_falsify` search on its own.


In [ ]:
# the network from the blackboard; the property is f(x) > 0
def f_toy(x):   # x: [B,2] -> [B]
    return 2 * F.relu(x[:, 0] - x[:, 1]) - F.relu(x[:, 0] + x[:, 1]) + 1.6

x0 = torch.tensor([[1.0, 0.5]]); eps, alpha = 0.4, 0.1
print("f(x0) =", round(f_toy(x0).item(), 2))

# gradient of f_toy with respect to the input
def grad_f(x):
    xr = x.clone().requires_grad_(True); f_toy(xr).sum().backward(); return xr.grad

# FGSM: one step, the whole budget at once
gr = grad_f(x0); x_fgsm = (x0 - eps * gr.sign()).clamp(0, 1)
print("FGSM, 1 step  : x' =", [round(v, 2) for v in x_fgsm[0].tolist()],
      " f =", round(f_toy(x_fgsm).item(), 2),
      "  (the local linear model predicted", round(f_toy(x0).item() - eps * gr.abs().sum().item(), 2), ")")

# PGD by hand: a step against the gradient, then clip onto C_p(x0, eps) and onto [0,1]
x = x0.clone()
print("\nstep   x_t            f(x_t)  gradient at x_{t-1}")
for t in range(1, 6):
    gr = grad_f(x)
    x = torch.min(torch.max(x - alpha * gr.sign(), x0 - eps), x0 + eps).clamp(0, 1)
    print(f"  {t}    {[round(v, 2) for v in x[0].tolist()]}   {f_toy(x).item():5.2f}    {[round(v, 1) for v in gr[0].tolist()]}")
print("minimum of f over the box: -0.20 at (0.9, 0.9)")

# and now the automatic search: random start inside C_p(x0, eps), 10 restarts
torch.manual_seed(0)       # the start is random: without a seed this number changes every run
bm, bd = pgd_falsify(lambda x: f_toy(x)[:, None], x0, None, eps, spec_score,
                     steps=30, restarts=10, early_stop=False)
print("\npgd_falsify (no early stop): smallest f =", round(bm.item(), 2),
      "at x =", [round(v, 2) for v in (x0 + bd)[0].tolist()])


**Questions.**

(i) FGSM stays at $f > 0$ even though the local linear model predicted a negative value. What
happened on the way?

(ii) At which step does the gradient change, and why? After how many steps does the class flip?

(iii) Why do we have to re-linearise at every step? How does this relate to the *piecewise
linearity* we saw in the lecture?

(iv) `pgd_falsify` returns about $-0.18$, while the true minimum over the box is $-0.20$ at
$(0.9, 0.9)$ — it finds *a* counterexample but not the deepest one. Why can it not reach that
corner? (*Hint:* from a random start, sign steps of size $\alpha$ can only ever land on a grid of
points.) Both numbers are $\le 0$, so the property is falsified either way: what does that tell you
about what an attack actually measures — the true minimum, or an upper bound on it?


### 5b. On MNIST


In [ ]:
ok = cnn(X).argmax(1) == y            # keep only the images classified correctly to begin with
Xc, yc = X[ok][:100], y[ok][:100]
# two budgets x two models x one or ten restarts
print("eps  model         restarts  falsified  margin (min .. median)")
for eps in (0.1, 0.2):
    for name, m in (("cnn", cnn), ("cnn_robust", cnn_robust)):
        for R in (1, 10):
            bm, _ = pgd_falsify(m, Xc, yc, eps, spec_class, restarts=R)
            print(f"{eps:<4} {name:<13} {R:>6}   {(bm <= 0).float().mean():>9.0%}   {bm.min():8.2f} .. {bm.median():6.2f}")


In [ ]:
# the images falsified at eps=0.1, before and after the perturbation
bm, bd = pgd_falsify(cnn, Xc, yc, 0.1, spec_class, restarts=10)
idx = (bm <= 0).nonzero()[:10, 0]
show(Xc[idx], yc[idx], cnn, n=len(idx), title="originals")
show((Xc + bd)[idx], yc[idx], cnn, n=len(idx), title="counterexamples found (eps=0.1)")


**What to take away (*"THE ASYMMETRY"*):** an attack that **succeeds** proves the model is broken;
an attack that **fails proves nothing**. PGD gives an **upper bound** on $y^\star$:
$\le 0$ → falsified, $> 0$ → we do not know. To claim the network *is* robust we need a **lower
bound** $\ge 0$ on $y^\star$ — that is, verification, and that is where the rest of the course goes.

**Questions:** how much do the restarts help (1 vs 10)? On `cnn_robust` at $\varepsilon = 0.1$ PGD
finds no counterexample at all, yet the smallest margin is small and positive: may we call it robust?


---
## 6. Exercise 2 — The smallest falsifying ε

**Break a classifier, and measure how far you got.** For each image we look for the **smallest
$\varepsilon$** at which PGD (the `pgd_falsify` of Section 5) finds a counterexample, searching over
a grid of values. Call it $\varepsilon^*(x)$. Then we compare `cnn` with `cnn_robust`.

**What to hand in:** nothing. Bring the notebook with the cells executed and the answers written in;
we discuss them together in the next class.

**Fill in the line marked `FILL_IN`.**


In [ ]:
eps_grid = [0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4]

# for every image, the first eps of the grid at which pgd_falsify finds a counterexample
def first_falsifying_eps(model, X, y, grid, restarts=5):
    """For each image: the first eps in the grid at which PGD finds a counterexample (inf if never)."""
    eps_star = torch.full((len(X),), float("inf"))
    # budgets in increasing order
    for eps in grid:
        bm, _ = pgd_falsify(model, X, y, eps, spec_class, steps=50, restarts=restarts)
        newly = (bm <= 0) & torch.isinf(eps_star)       # falsified here for the FIRST time
        eps_star[FILL_IN_1] = FILL_IN_2                          # TODO: record eps for the 'newly' images
    return eps_star


In [ ]:
# --- Sanity checks. These must pass before you trust the plot. ---
# 10 images and 2 restarts, to be fast
_Xs, _ys = X[:10], y[:10]
_es = first_falsifying_eps(cnn, _Xs, _ys, eps_grid, restarts=2)

assert _es.shape == (10,), "first_falsifying_eps must return one value per image"
assert torch.isfinite(_es).any(), \
    "no image falsified even at eps=0.4: are you writing into eps_star at all?"
assert all(min(abs(v - g) for g in eps_grid) < 1e-6 for v in _es[torch.isfinite(_es)].tolist()), \
    "every finite eps* must be one of the values in the grid"
_frac = [(_es <= g).float().mean().item() for g in eps_grid]
assert all(a <= b + 1e-9 for a, b in zip(_frac, _frac[1:])), \
    "eps*(x) must record the FIRST eps that works, so the curve can only go up: check the mask"
print("exercise sanity checks passed.")


In [ ]:
ok2 = (cnn(X).argmax(1) == y) & (cnn_robust(X).argmax(1) == y)   # correctly classified by both
Xe, ye = X[ok2][:50], y[ok2][:50]
# eps* for both models
eps_star = {name: first_falsifying_eps(m, Xe, ye, eps_grid)
            for name, m in (("cnn", cnn), ("cnn_robust", cnn_robust))}

# how many images fell, and the median eps* among them
for name, e in eps_star.items():
    done = e[torch.isfinite(e)]
    med = done.median().item() if len(done) else float("nan")
    print(f"{name:<11} falsified: {len(done)}/{len(e)}   median eps* among those: {med:.2f}")

for name, e in eps_star.items():       # fraction of images already falsified as eps grows
    plt.step(eps_grid, [(e <= g).float().mean().item() for g in eps_grid],
             where="post", marker="o", label=name)
plt.xlabel("epsilon"); plt.ylabel("fraction of images falsified"); plt.legend(); plt.grid(); plt.show()


### Questions (write your answers below, 2–4 lines each)

1. **Measure.** What is the median $\varepsilon^*$ for the two networks? How many images of the
   robust network survive all the way to $\varepsilon = 0.4$?
2. **What kind of number is it?** Is $\varepsilon^*(x)$ an *upper* or a *lower* bound on the true
   robustness radius of $x$ (the smallest $\varepsilon$ for which a counterexample exists)? What can
   you conclude for an $\varepsilon < \varepsilon^*$? *(Hint: "THE ASYMMETRY" and "Measuring
   robustness" in the slides — and question (iv) of Section 5a.)*
3. **How much do you trust a single attack?** Re-run the computation for `cnn_robust` with
   `restarts=1` and compare with `restarts=5`. What changes? What does that imply for anyone
   reporting the "robust accuracy" of a model?
4. **For the discussion in class.** If you had to *guarantee* to a customer that `cnn_robust` is
   robust at $\varepsilon = 0.1$ on a given image, would what you have done here be enough? What
   else would you need?


**Answer 1:**

**Answer 2:**

**Answer 3:**

**Answer 4:**


---
## Extra: targeted attack

Not covered in class, left here for whoever is curious. The targeted version of the attack, as in the
slides (*"FGSM — one step towards a class you choose"*): the same loop as `pgd`, but we **descend**
the loss of a target class $t$ instead of climbing the loss of the true class $y$,

$$x_{k} = \mathrm{clip}_C\big(x_{k-1} - \alpha \cdot \mathrm{sign}(\nabla_x \mathrm{loss}_t(x_{k-1}))\big)$$

and the attack succeeds when the network predicts exactly $t$. The target can be different for every image.

In [ ]:
# the same loop as pgd, but DESCENDING the loss of the target t (note the minus)
def pgd_targeted(model, X, t, eps, alpha=None, iters=40):
    """t: one target label per image. Returns eta such that model(X + eta) should predict t."""
    alpha = alpha or eps / 4
    eta = torch.zeros_like(X, requires_grad=True)
    for _ in range(iters):
        loss = nn.CrossEntropyLoss()(model(X + eta), t)
        loss.backward()
        d = (eta - alpha * eta.grad.sign()).clamp(-eps, eps)
        eta = ((X + d).clamp(0, 1) - X).detach().requires_grad_(True)
    return eta.detach()

# fraction of images on which the network predicts exactly the target
def hit_rate(model, X, t):
    with torch.no_grad(): return (model(X).argmax(1) == t).float().mean().item()

# every image towards the same class (3); the ones that are already 3 are dropped
eps = 0.2
target = 3
mask = y != target
Xt, yt = X[mask], y[mask]
t = torch.full_like(yt, target)
Xa = Xt + pgd_targeted(cnn, Xt, t, eps)
print("all images -> %d          : %.1f%% classified as the target" % (target, 100 * hit_rate(cnn, Xa, t)))
show(Xa, yt, cnn, title=f"targeted -> {target}, eps={eps}")

# a different target for every image: the next digit
t = (y + 1) % 10
Xa = X + pgd_targeted(cnn, X, t, eps)
print("each image -> its own y+1 : %.1f%% classified as the target" % (100 * hit_rate(cnn, Xa, t)))
show(Xa, y, cnn, title=f"targeted -> y+1, eps={eps}")

---
## Further reading

**For the attacks themselves there is no textbook**, which is why the lecture notes exist. The
primary sources, one topic each:

- Goodfellow, Shlens & Szegedy, *Explaining and Harnessing Adversarial Examples*, ICLR 2015 — FGSM,
  and the linearity argument for why adversarial examples exist at all.
- Madry, Makelov, Schmidt, Tsipras & Vladu, *Towards Deep Learning Models Resistant to Adversarial
  Attacks*, ICLR 2018 — PGD, and adversarial training as the min-max problem. This is **the**
  reference for adversarial training; there is no book that does the job better.
- Carlini & Wagner, *Towards Evaluating the Robustness of Neural Networks*, IEEE S&P 2017 — the
  change of variable, the objective design, and the $\tau$ schedule of Section 4 of the slides.
- Athalye, Carlini & Wagner, *Obfuscated Gradients Give a False Sense of Security*, ICML 2018 — the
  paper behind the sanity checks in Section 4b, and behind THE TRAP.
- Tsipras, Santurkar, Engstrom, Turner & Madry, *Robustness May Be at Odds with Accuracy*, ICLR 2019
  — the clean-accuracy price you measure yourself in Section 4.
- Bai, Luo, Zhao, Wen & Wang, *Recent Advances in Adversarial Training for Adversarial Robustness*,
  IJCAI 2021 — one survey, if you want the landscape rather than a single method.

**A note if you read more widely.** Much of this literature writes the attack as a *maximisation of
the loss* rather than a *minimisation of the margin*, and then says an attack gives a "lower bound".
That is the same statement as ours with the sign flipped: a lower bound on the worst-case loss is an
upper bound on $y^\star$. The Kolter & Madry tutorial (https://adversarial-ml-tutorial.org/) is
written that way; it is worth reading for a second viewpoint, but keep the translation in mind, and
do not run its code — it was written for an old version of PyTorch.
